# QTran 三数据集公平比较实验

本 Notebook 对 WM-811K、MixedWM38 和 Carinthia 分别训练五量子投影 QTran 及三个参数匹配基线。三个数据集互不混合、检查点互不共用；模型选择只使用验证集，测试集仅在最佳验证检查点冻结后评价。

## 0. 使用顺序

1. 先运行路径与原始数据检查。
2. 三个缓存单元分别运行一次。
3. 用 `DATASET_TO_RUN` 每次选择一个数据集。
4. 先执行审计和单种子试跑，确认后再运行五种子。
5. 三个数据集全部完成后再运行总表、统计和论文绘图单元。

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_wm811k.py').exists() and (PROJECT_DIR / 'QCS' / 'qcs_wm811k.py').exists():
    PROJECT_DIR = PROJECT_DIR / 'QCS'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from qcs_core import ExperimentConfig, environment_report, summarize_results
from qcs_datasets import (
    class_distribution, load_dataset_cache, make_dataset_split,
    prepare_carinthia_cache, prepare_mixedwm38_cache,
    prepare_wm811k_cache, split_distribution,
)
from qcs_multidataset import (
    dataset_audit, evaluate_noise_suite, paired_model_comparisons,
    plot_cross_dataset_results, run_comparison_suite,
)

print('PROJECT_DIR:', PROJECT_DIR)
print(environment_report())

In [ ]:
RAW_PATHS = {
    'wm811k': PROJECT_DIR / 'data' / 'LSWMD.pkl',
    'mixedwm38': PROJECT_DIR / 'data' / 'raw' / 'mixedwm38' / 'Wafer_Map_Datasets.npz',
    'carinthia': PROJECT_DIR / 'data' / 'raw' / 'carinthia' / 'data.zip',
}
CACHE_PATHS = {
    'wm811k': PROJECT_DIR / 'data_cache' / 'wm811k_labeled_32.npz',
    'mixedwm38': PROJECT_DIR / 'data_cache' / 'mixedwm38_32.npz',
    'carinthia': PROJECT_DIR / 'data_cache' / 'carinthia_32.npz',
}
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'three_datasets_five_projection'

path_rows = []
for name, path in RAW_PATHS.items():
    path_rows.append({
        'dataset': name, 'path': str(path), 'exists': path.exists(),
        'size_gb': path.stat().st_size / 1024**3 if path.exists() else None,
    })
display(pd.DataFrame(path_rows).round(3))
assert all(path.exists() for path in RAW_PATHS.values()), '存在缺失的原始数据文件'

## 1. 分别生成缓存

原始文件只读。已有缓存时会立即返回；只有明确需要重建时才将 `force` 改为 `True`。

In [ ]:
prepare_wm811k_cache(
    RAW_PATHS['wm811k'], CACHE_PATHS['wm811k'], image_size=32, force=False
)

In [ ]:
prepare_mixedwm38_cache(
    RAW_PATHS['mixedwm38'], CACHE_PATHS['mixedwm38'], image_size=32, force=False
)

In [ ]:
prepare_carinthia_cache(
    RAW_PATHS['carinthia'], CACHE_PATHS['carinthia'], image_size=32, force=False
)

## 2. 固定论文配置

该配置对四种模型完全一致。数据集适配器只会调整输入通道数和输出类别数。正式比较前不要根据测试结果修改配置。

In [ ]:
BASE_CONFIG = replace(
    ExperimentConfig.publication(),
    quantum_projection_mode='five',
    epochs=60,
    patience=10,
    batch_size=64,
    sampler_power=0.5,
    quantum_init_scale=0.1,
    train_cap_per_class=2000,
    eval_cap_per_class=None,
    num_workers=0,
)
SEEDS = (42, 52, 62, 72, 82)
SPLIT_SEED = 2026

# 每次只运行一个数据集：wm811k / mixedwm38 / carinthia
DATASET_TO_RUN = 'mixedwm38'
CACHE_PATH = CACHE_PATHS[DATASET_TO_RUN]
print(DATASET_TO_RUN, CACHE_PATH)

## 3. 数据、划分、参数量与梯度审计

In [ ]:
bundle = load_dataset_cache(CACHE_PATH)
split = make_dataset_split(bundle, seed=SPLIT_SEED)
print(bundle.describe())
display(class_distribution(bundle).round(3))
display(split_distribution(bundle, split).pivot(index='label', columns='split', values='count'))

audit = dataset_audit(
    CACHE_PATH, BASE_CONFIG, split_seed=SPLIT_SEED, run_gradient_test=True
)
display(audit['parameter_audit'].round(3))
display(audit['gradient_test'].round(3))

## 4. 可选：单种子短试跑

第一次接入某个数据集时先运行本单元。结果仅用于排错，不进入论文。

In [ ]:
SMOKE_CONFIG = replace(
    BASE_CONFIG, epochs=2, patience=2, train_cap_per_class=16, eval_cap_per_class=16
)
smoke_results, _, _ = run_comparison_suite(
    cache_path=CACHE_PATH,
    base_config=SMOKE_CONFIG,
    seeds=(42,),
    split_seed=SPLIT_SEED,
    artifact_dir=PROJECT_DIR / 'artifacts' / 'smoke_three_datasets',
    resume=True,
)
display(smoke_results.round(3))

## 5. 正式五种子比较

该单元可断点续跑。建议依次完成 `mixedwm38`、`carinthia`，最后在需要时用相同新管线复核 `wm811k`。

In [ ]:
results, histories, confusion_matrices = run_comparison_suite(
    cache_path=CACHE_PATH,
    base_config=BASE_CONFIG,
    seeds=SEEDS,
    split_seed=SPLIT_SEED,
    artifact_dir=ARTIFACT_ROOT,
    resume=True,
)
display(results.round(3))
display(summarize_results(results).round(3))

## 6. 三数据集完成后汇总

若某个数据集尚未完成，本单元会显示缺失项，不会伪造或填补结果。

In [ ]:
frames = []
missing = []
for dataset_name in ('wm811k', 'mixedwm38', 'carinthia'):
    result_file = ARTIFACT_ROOT / dataset_name / 'comparison_results.csv'
    if result_file.exists():
        frame = pd.read_csv(result_file)
        expected = len(SEEDS) * 4
        if len(frame) == expected:
            frames.append(frame)
        else:
            missing.append(f'{dataset_name}: {len(frame)}/{expected} jobs')
    else:
        missing.append(f'{dataset_name}: no result file')

print('Missing:', missing if missing else 'none')
if not missing:
    all_results = pd.concat(frames, ignore_index=True)
    all_results.to_csv(ARTIFACT_ROOT / 'three_dataset_results.csv', index=False)
    display(all_results.groupby(['dataset', 'model'])[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(3))

In [ ]:
if not missing:
    paired = paired_model_comparisons(all_results, metric='macro_f1')
    paired.to_csv(ARTIFACT_ROOT / 'paired_macro_f1.csv', index=False)
    display(paired.round(3))

In [ ]:
if not missing:
    figure = plot_cross_dataset_results(all_results, metric='macro_f1')
    figure_dir = PROJECT_DIR / 'paper' / 'figures'
    figure_dir.mkdir(parents=True, exist_ok=True)
    figure.savefig(figure_dir / 'three_dataset_macro_f1.png', bbox_inches='tight', dpi=300)
    figure.savefig(figure_dir / 'three_dataset_macro_f1.pdf', bbox_inches='tight')
    plt.show()

## 7. 可选：已保存检查点的噪声鲁棒性

晶圆图噪声表示缺陷位翻转概率；Carinthia 噪声表示归一化灰度图上的高斯噪声标准差。两种噪声不能直接解释为同一种物理扰动。

In [ ]:
noise_results = evaluate_noise_suite(
    cache_path=CACHE_PATH,
    artifact_dir=ARTIFACT_ROOT,
    seeds=SEEDS,
    noise_levels=(0.00, 0.01, 0.03, 0.05, 0.10),
    split_seed=SPLIT_SEED,
)
display(noise_results.round(3))